In [30]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, precision_recall_curve, roc_curve, auc

from utils.model_utils import validate, train, evaluate, get_tensor_dataset, get_data_loader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [31]:
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")
val_df = pd.read_csv("data/val.csv")

In [32]:
label = 'label'

train_dataset = get_tensor_dataset(train_df, label)
test_dataset = get_tensor_dataset(test_df, label)
val_dataset = get_tensor_dataset(val_df, label)

print(train_dataset)
print(test_dataset)
print(val_dataset)

In [33]:
BATCH_SIZE = 32
train_loader = get_data_loader(train_dataset, BATCH_SIZE)
test_loader = get_data_loader(test_dataset, BATCH_SIZE)
val_loader = get_data_loader(val_dataset, BATCH_SIZE)

In [34]:
model = nn.Sequential(
    nn.Linear(1024, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(0.4),

    nn.Linear(256, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(0.2),

    nn.Linear(64, 1)
)
model.to(device)
print(model)

Sequential(
  (0): Linear(in_features=1024, out_features=256, bias=True)
  (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): Dropout(p=0.4, inplace=False)
  (4): Linear(in_features=256, out_features=64, bias=True)
  (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (6): ReLU()
  (7): Dropout(p=0.2, inplace=False)
  (8): Linear(in_features=64, out_features=1, bias=True)
)


In [35]:
model.eval()
all_probs = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch).squeeze()
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(y_batch.numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)
thresholds = np.arange(0.1, 0.9, 0.01)

f1_scores = [f1_score(all_labels, (all_probs >= t).astype(int), average='macro') for t in
             thresholds]

best_threshold = thresholds[np.argmax(f1_scores)]
best_f1 = max(f1_scores)
print(f"Best threshold: {best_threshold:.2f} → F1: {best_f1:.4f}")

Best threshold: 0.53 → F1: 0.4674


In [37]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
THRESHOLD = best_threshold

In [ ]:
best_loss = float('inf')
patience = 10
counter = 0

for epoch in range(50):
    print("Epoch:", epoch)
    train(
        model=model,
        loss_function=criterion,
        threshold=THRESHOLD,
        optimizer=optimizer,
        train_loader=train_loader,
        device=device
    )
    val_loss = validate(
        model=model,
        loss_function=criterion,
        threshold=THRESHOLD,
        val_loader=val_loader,
        device=device
    )
    print("-" * 40)

    if val_loss < best_loss:
        best_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), "../model_repository/pneumonia_model_phase_2.pth")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break

Epoch: 0
Train - Loss: 0.7618 Accuracy: 0.7135
Valid - Loss: 0.7060 Accuracy: 0.7315
----------------------------------------
Epoch: 1
Train - Loss: 0.6726 Accuracy: 0.7651
Valid - Loss: 0.6755 Accuracy: 0.7728
----------------------------------------
Epoch: 2
Train - Loss: 0.6184 Accuracy: 0.7869
Valid - Loss: 0.6720 Accuracy: 0.7573
----------------------------------------
Epoch: 3
Train - Loss: 0.5701 Accuracy: 0.8079
Valid - Loss: 0.6696 Accuracy: 0.7849
----------------------------------------
Epoch: 4
Train - Loss: 0.5450 Accuracy: 0.8142
Valid - Loss: 0.6808 Accuracy: 0.7676
----------------------------------------
Epoch: 5
Train - Loss: 0.5032 Accuracy: 0.8278
Valid - Loss: 0.6882 Accuracy: 0.7642
----------------------------------------
Epoch: 6
Train - Loss: 0.4636 Accuracy: 0.8414
Valid - Loss: 0.7438 Accuracy: 0.7556
----------------------------------------
Epoch: 7
Train - Loss: 0.4108 Accuracy: 0.8551
Valid - Loss: 0.7566 Accuracy: 0.7315
---------------------------------

In [ ]:
evaluate(model=model,
         threshold=THRESHOLD,
         data_loader=test_loader,
         device=device,
         labels=["BACTERIAL", "VIRAL"])